<a href="https://colab.research.google.com/github/alter-mix-dev/wasapcloudapi/blob/feat%2Fpedidos/AGENTE_PEDIDOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# BLOCK 1: PREPARACIÓN DEL ENTORNO Y CREDENCIALES
#En este bloque instalamos todas las librerías necesarias de Python y del sistema,
# importamos los módulos globales y cargamos de forma segura tus credenciales actualizadas desde Colab Secrets

In [2]:
# 1.1 INSTALACIÓN DE DEPENDENCIAS
# Instalamos componentes del sistema y librerías de Python requeridas en un solo paso
!apt-get update -qq
!apt-get install -y zstd -qq  # Requerido por Ollama
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama fastapi uvicorn pyngrok requests pytz --quiet

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# 1.2 IMPORTS GLOBALES Y CONFIGURACIÓN HORARIA
# ==============================================================================
import os
import time
import threading
import subprocess
import requests
from datetime import datetime
import pytz

import ollama
import uvicorn
from fastapi import FastAPI, Request, BackgroundTasks
from pyngrok import ngrok
from google.colab import userdata


In [4]:
# 1.3 CARGA DE CREDENCIALES (COLAB SECRETS)
# ==============================================================================
VERIFY_TOKEN = "mi_token"  # Debe coincidir con el configurado en Meta for Developers
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')  # Tu nuevo token de 2 meses
PHONE_ID = userdata.get('PHONE_NUMBER_ID')
WABA_ID = "1061287516789190"  # ID de la cuenta de WhatsApp Business

print("✅ Entorno preparado e imports cargados.")
print("✅ Credenciales leídas desde Secrets correctamente.")

✅ Entorno preparado e imports cargados.
✅ Credenciales leídas desde Secrets correctamente.


In [5]:
#BLOCK 2: BACKEND DE IA (OLLAMA & MODELO)
#Este bloque se encarga de inicializar el servidor local de Ollama en la máquina virtual de Colab en segundo plano
# y descargar el modelo Llama 3.1.

In [6]:
# 2.1 ARRANQUE DEL SERVIDOR OLLAMA (tarda como 2 minutos)
proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Tiempo de espera para levantar el servicio local

# Descarga del modelo específico
!ollama pull llama3.1:8b
print("✅ Servidor de Ollama y Llama 3.1 listos.")


✅ Servidor de Ollama y Llama 3.1 listos.


In [7]:
# 2.2 INSTALAR Y LEVANTAR LLAMA STACK
!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

In [8]:
# 2.3. Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado
import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


In [9]:
#BLOCK 3: LÓGICA DE NEGOCIO Y MODELO DE CONTEXTO
#Definimos la lógica del asistente del gimnasio. El modelo ahora recibe inyección dinámica del día y la hora de México
# para responder de manera coherente si el usuario pregunta por horarios o membresías.

In [9]:
# 3.1 BASE DE DATOS EN MEMORIA
# ==============================================================================
mensajes_recibidos = []
usuarios_conocidos = set()  # Almacena los números que ya recibieron el saludo inicial

In [10]:
# 3.2 HELPERS Y GENERACIÓN DE RESPUESTA CON CONTEXTO TEMPORAL
# ==============================================================================
def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

def generar_respuesta(texto_usuario):
    # Inyección horaria de la CDMX para que el bot conozca el tiempo real
    zona_horaria = pytz.timezone('America/Mexico_City')
    ahora = datetime.now(zona_horaria)
    fecha_actual = ahora.strftime("%A, %d de %B de %Y")
    hora_actual = ahora.strftime("%H:%M")

    prompt = (
        f"Eres el asistente de atención a clientes de un gimnasio. "
        f"Hoy es {fecha_actual} y la hora actual es {hora_actual}. "
        f"Responde de forma breve (2-3 líneas) a la siguiente pregunta sobre horarios o membresías:\n\n"
        + texto_usuario
    )

    respuesta = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content": prompt}])
    return respuesta["message"]["content"]

print("✅ Helpers de lógica de negocio y generación de prompts configurados.")

✅ Helpers de lógica de negocio y generación de prompts configurados.


In [17]:
#BLOCK 4: DEFINICIÓN DEL SERVIDOR WEBHOOK (FASTAPI)
#Configuramos nuestra aplicación FastAPI con los endpoints requeridos por Meta. Procesamos los mensajes
#entrantes en segundo plano (BackgroundTasks) para mandar una confirmación HTTP inmediata a WhatsApp y evitar duplicidades.
_ = '''
# 4.1 ENDPOINTS DE FASTAPI (CORREGIDO Y CONFIGURADO PARA PRODUCCIÓN)
# ==============================================================================
app = FastAPI()

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        # Extraemos de forma segura navegando las listas del JSON de Meta
        if "entry" in body and len(body["entry"]) > 0:
            entry = body["entry"][0]
            if "changes" in entry and len(entry["changes"]) > 0:
                change = entry["changes"][0]
                value = change.get("value", {})

                # IMPORTANTÍSIMO: Validar que sea un mensaje de un cliente y no una confirmación de Meta
                if "messages" in value and len(value["messages"]) > 0:
                    datos_mensaje = value["messages"][0]

                    # Filtramos únicamente los mensajes de texto entrantes
                    if datos_mensaje.get("type") == "text":
                        numero = normalizar_numero_mx(datos_mensaje["from"])
                        texto = datos_mensaje["text"]["body"]

                        print(f"📥 MENSAJE RECIBIDO -> De: {numero} | Texto: '{texto}'")

                        # Mandamos la tarea al hilo secundario de Ollama
                        background_tasks.add_task(procesar_y_responder, numero, texto)
                    else:
                        print("ℹ️ Mensaje ignorado: No es tipo texto (ej. imagen, audio o interacción).")

                # Si Meta solo nos está mandando un estatus de lectura/entrega, lo registramos sin responder
                elif "statuses" in value:
                    print(f"📈 Notificación de Meta: Mensaje {value['statuses'][0].get('status')}.")

    except Exception as e:
        print(f"❌ Error crítico procesando el JSON de Meta: {e}")

    return {"status": "ok"}

# ==============================================================================
# 4.2 ACCIONES DEL WEBHOOK (CON DIAGNÓSTICO DE ERRORES HTTP)
# ==============================================================================
def enviar_respuesta(numero, texto):
    url = f"https://facebook.com/{PHONE_ID}/messages"
    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json"
    }
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    # Realizamos la petición capturando la respuesta de Meta
    respuesta_meta = requests.post(url, headers=headers, json=payload)

    if respuesta_meta.status_code == 200:
        print(f"📤 MENSAJE ENVIADO CON ÉXITO a {numero}")
    else:
        print(f"❌ Error {respuesta_meta.status_code} al enviar mensaje a Meta.")
        print(f"Detalle técnico de Meta: {respuesta_meta.text}")

def procesar_y_responder(numero, texto):
    try:
        if numero not in usuarios_conocidos:
            usuarios_conocidos.add(numero)
            enviar_respuesta(numero, "¡Hola! Soy el asistente del gimnasio. ¿En qué puedo ayudarte?")
        else:
            print(f"🤖 Consultando a Ollama (Llama 3.1) para contestarle a {numero}...")
            respuesta_ia = generar_respuesta(texto)
            print(f"✍️ Respuesta generada por la IA: '{respuesta_ia}'")
            enviar_respuesta(numero, respuesta_ia)
    except Exception as e:
        print(f"❌ Fallo en el subproceso de la IA (procesar_y_responder): {e}")

print("✅ Servidor Webhook robusto y actualizado.")
'''

_IncompleteInputError: incomplete input (4119057204.py, line 93)

In [18]:
#_ = '''
# 4.1 ENDPOINTS DE FASTAPI
# ==============================================================================
app = FastAPI()

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}  # Meta recibe el ACK inmediato


In [19]:
#_ = '''
# 4.2 ACCIONES DEL WEBHOOK
# ==============================================================================
def enviar_respuesta(numero, texto):
    url = f"https://facebook.com{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }
    requests.post(url, headers=headers, json=payload)

def procesar_y_responder(numero, texto):
    if numero not in usuarios_conocidos:
        usuarios_conocidos.add(numero)
        enviar_respuesta(numero, "¡Hola! Soy el asistente del gimnasio. ¿En qué puedo ayudarte?")
    else:
        respuesta = generar_respuesta(texto)
        enviar_respuesta(numero, respuesta)

print("✅ Servidor Webhook declarado.")

✅ Servidor Webhook declarado.


In [15]:
#BLOCK 5: DESPLIEGUE, TÚNEL NGROK Y PRUEBAS
#Este bloque expone tu servidor local al internet público por medio de un túnel ngrok y lanza las simulaciones de prueba.
#Incluye el código muerto de tus diagnósticos y la prueba explícita para tu nuevo token.

In [20]:
# 5.1 ACTIVACIÓN DE TÚNEL Y SERVIDOR
# ==============================================================================
ngrok.kill()  # Limpiamos cualquier túnel previo trabado
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)

print("🔗 URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook\n")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()
time.sleep(2)  # Dar tiempo para asegurar el arranque


INFO:     Started server process [19442]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 URL a configurar como Callback URL en el dashboard de Meta:
https://mutate-stylus-facedown.ngrok-free.dev/webhook



In [21]:
# 5.2 SIMULACIÓN DE FLUJO COMPLETO (LOCAL)
# ==============================================================================
payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "Hola"}}
    ]}}]}]
}
r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("🧪 Simulación de primer mensaje (Saludo):", r1.status_code, r1.text)


INFO:     127.0.0.1:39186 - "POST /webhook HTTP/1.1" 200 OK
🧪 Simulación de primer mensaje (Saludo): 200 {"status":"ok"}


In [18]:
# 5.3 PRUEBA EXCLUSIVA PARA EL TOKEN DE 2 MESES (CONEXIÓN CON META)
# ==============================================================================
print("\n🔍 --- PROBANDO VALIDEZ DE TU NUEVO TOKEN DE 2 MESES ---")
# Intentamos vincular la app con el WABA ID usando el nuevo token de secrets
r_token = requests.post(
    f"https://graph.facebook.com/{1061287516789190}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)

if r_token.status_code == 200:
    print("✅ ¡ÉXITO! Meta aceptó el token correctamente. Código 200 OK.")
    print("Detalle:", r_token.text)
else:
    print(f"❌ ERROR {r_token.status_code}: El token guardado en Secrets sigue dando problemas.")
    print("Respuesta de Meta:", r_token.text)


🔍 --- PROBANDO VALIDEZ DE TU NUEVO TOKEN DE 2 MESES ---


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 239, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/socket.py", line 983, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno -2] Name or service not known

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/ur

✅ ¡ÉXITO! Meta aceptó el token correctamente. Código 200 OK.
Detalle: {"success":true}


In [22]:
# 5.4 CÓDIGO MUERTO DE PRUEBAS ADICIONALES Y DIAGNÓSTICOS ANTERIORES
# ==============================================================================
_ = '''
payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "¿A qué hora abren los sábados?"}}
    ]}}]}]
}
r2 = requests.post("http://127.0.0", json=payload_2)
print("Segundo mensaje:", r2.status_code, r2.text)
'''

_ = '''
if 'mensajes_recibidos' not in locals(): mensajes_recibidos = []
if 'respuestas_enviadas' not in locals(): respuestas_enviadas = 0
if 'errores_procesamiento' not in locals(): errores_procesamiento = 0
if 'conversaciones_por_numero' not in locals(): conversaciones_por_numero = {}

print("--- DIAGNÓSTICO DEL BOT ---")
print("Mensajes recibidos:", len(mensajes_recibidos))
print("Respuestas enviadas:", respuestas_enviadas)
print("Errores:", errores_procesamiento)
try: print("Llama Stack vivo:", proceso_llama_stack.poll() is None)
except NameError: print("Llama Stack vivo: Error (Variable no definida)")
try: print("Servidor vivo:", hilo.is_alive())
except NameError: print("Servidor vivo: Error (Variable no definida)")
'''

_ = '''
url = tunel_publico.public_url + "/webhook"
r = requests.get(url, params={"hub.verify_token": VERIFY_TOKEN, "hub.challenge": "12345"})
print("Prueba Túnel:", r.status_code, r.text)
'''

In [23]:
import socket

def verificar_puerto(puerto, nombre_servicio):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(2)
    resultado = sock.connect_ex(('127.0.0.1', puerto))
    if resultado == 0:
        print(f"🟢 PUERTO {puerto} [{nombre_servicio}]: ACTIVADO y escuchando correctamente.")
    else:
        print(f"🔴 PUERTO {puerto} [{nombre_servicio}]: CERRADO o inactivo. (Código de error: {resultado})")
    sock.close()

print("--- AUDITORÍA DE RED EN TIEMPO REAL ---")
verificar_puerto(11434, "Servidor Interno de Ollama")
verificar_puerto(8000, "Tu Servidor Webhook FastAPI")

--- AUDITORÍA DE RED EN TIEMPO REAL ---
🟢 PUERTO 11434 [Servidor Interno de Ollama]: ACTIVADO y escuchando correctamente.
🟢 PUERTO 8000 [Tu Servidor Webhook FastAPI]: ACTIVADO y escuchando correctamente.
